# Fase 3 — Notebook 02: Análise Exploratória (EDA)

**Entrada:** `banco_de_dados/Base_ELSI_Bruta_Censo2022.csv` (produzido pelo Notebook 01).

**Objetivo:** caracterizar a base filtrada dos 70 municípios ELSI antes de qualquer
cálculo de IVS — descrever distribuições, detectar outliers, avaliar dados faltantes
e estudar a estrutura de correlação entre as 7 variáveis-componente do IVS. Esta EDA
alimenta a **Tabela 1** do artigo científico.

**Referência metodológica:**
- `docs/Cálculo IVS2012.docx` — regras operacionais do IVS-BH (denominador,
  `Dados_sig`, tratamento de sigilo).
- `docs/guia_analises.docx` — framework FIOCRUZ de EDA: medidas de tendência
  central (média / mediana / quantis), dispersão (DP / IQR / CV), gráficos
  (histograma, boxplot, matriz de correlação) e análise de missing.

**Decisão metodológica resolvida nesta etapa:**
Conforme `Cálculo IVS2012.docx`, seção *Tratamento dos dados*: "achamos melhor
considerar o número de responsáveis como número total de domicílios do setor".
Portanto o denominador adotado para os indicadores de saneamento é **V01042 (Total
de Responsáveis)**, não V00001. Isto resolve a inconsistência apontada no
`DIAGNOSTICO_COMPLETO_PROJETO.md` (Problema #3).

**Roteiro:**
1. Imports e carregamento.
2. Tipagem e tratamento de sigilo (`X` → `NaN`).
3. Classificação `Dados_sig` e filtro dos setores `OK`.
4. Cálculo das 7 proporções brutas (sem normalização).
5. Descritivas globais.
6. Descritivas por município.
7. Descritivas por região.
8. Distribuições — histogramas.
9. Distribuições — boxplots por região.
10. Análise de outliers (regra IQR).
11. Análise de dados faltantes.
12. Matriz de correlação (Pearson e Spearman).
13. Exportação dos artefatos.

## 1. Imports e carregamento da base filtrada

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)


def _find_project_root():
    """Detecta a raiz do projeto (independente de onde o Jupyter inicia o kernel)."""
    cwd = Path.cwd().resolve()
    for d in [cwd, *cwd.parents]:
        if (d / 'requirements.txt').is_file() and (d / 'dados').is_dir() and (d / 'docs').is_dir():
            return d
    raise RuntimeError(f'Raiz do projeto não encontrada a partir de: {cwd}')

ROOT = _find_project_root()
CAMINHO_BD  = str(ROOT / 'banco_de_dados') + os.sep
CAMINHO_EDA = str(ROOT / 'banco_de_dados' / 'eda') + os.sep
CAMINHO_FIG = str(ROOT / 'banco_de_dados' / 'eda' / 'figuras') + os.sep
os.makedirs(CAMINHO_EDA, exist_ok=True)
os.makedirs(CAMINHO_FIG, exist_ok=True)
print(f'Raiz do projeto: {ROOT}')

df = pd.read_csv(CAMINHO_BD + 'Base_ELSI_Bruta_Censo2022.csv', sep=';', dtype=str)
print(f'Base carregada: {len(df):,} setores × {len(df.columns)} colunas')
print(f'Municípios distintos: {df["CD_MUN"].nunique()}')
print(f'UFs distintas: {df["CD_UF"].nunique()}')

ModuleNotFoundError: No module named 'matplotlib'

## 2. Tipagem e tratamento de sigilo

O IBGE marca células sigilosas com `X`. Para a EDA, convertemos `X` em `NaN` (as
estatísticas descritivas pulam essas observações). A classificação `Dados_sig` da
próxima célula decide quais setores ficam fora da análise.

In [ ]:
COLS_TEXTO = ['CD_SETOR', 'CD_UF', 'CD_MUN', 'NM_MUN', 'NM_BAIRRO',
              'SITUACAO', 'Moradia_Predominante']
cols_num = [c for c in df.columns if c not in COLS_TEXTO]

# Marca sigilo ('X') como NaN.
df[cols_num] = df[cols_num].replace({'X': None, 'x': None})

# Algumas colunas (notadamente V06004 — rendimento médio) usam vírgula como
# separador decimal ('2453,03' em vez de '2453.03'). Trocamos antes de
# converter para numérico, senão pd.to_numeric devolve NaN para a maioria.
df[cols_num] = df[cols_num].apply(
    lambda c: c.astype(str).str.replace(',', '.', regex=False)
)
df[cols_num] = df[cols_num].apply(pd.to_numeric, errors='coerce')

print('Tipos após conversão:')
print(df[cols_num[:5]].dtypes.to_string())
print(f"\nTotal de células nulas (incluindo sigilo): {df[cols_num].isna().sum().sum():,}")
print(f"V06004 (renda média) — válidos: {df['V06004'].notna().sum():,}/{len(df):,}")

## 3. Classificação `Dados_sig` e filtro de setores elegíveis

Implementa as regras do `Cálculo IVS2012.docx`:
- **SIGILOSO** — alguma das variáveis-base (`v0001`, `V00001`, `V01042`) é sigilo (`NaN`).
- **COLETIVO** — 100% dos domicílios são coletivos: `dom_col = V01042 − V00001 − V00002`,
  `% coletivos = dom_col / V01042 × 100`.
- **ZERADO** — `v0001 = 0`.
- **OK** — caso contrário; participa das análises.

In [ ]:
dom_col = (df['V01042'] - df['V00001'] - df['V00002']).clip(lower=0)
perc_col = np.where(df['V01042'] > 0, dom_col / df['V01042'] * 100, 0)

cond_sig   = df[['v0001', 'V00001', 'V01042']].isna().any(axis=1)
cond_col   = perc_col >= 100
cond_zerado = df['v0001'].fillna(-1) == 0

df['Dados_sig'] = np.select(
    [cond_sig, cond_col, cond_zerado],
    ['SIGILOSO', 'COLETIVO', 'ZERADO'],
    default='OK',
)

resumo = df['Dados_sig'].value_counts().rename('n_setores').to_frame()
resumo['pct'] = (resumo['n_setores'] / len(df) * 100).round(2)
print('Elegibilidade dos setores (70 municípios ELSI):')
print(resumo.to_string())

df_ok = df[df['Dados_sig'] == 'OK'].copy()
print(f'\nSetores OK para análise: {len(df_ok):,}')
print(f'Municípios representados: {df_ok["CD_MUN"].nunique()}')

## 4. Cálculo das 7 proporções brutas (sem normalização)

Para a EDA usamos as **proporções brutas** dos componentes do IVS — a normalização
min-max só faz sentido depois, na construção do índice. Todos os denominadores
seguem o `Cálculo IVS2012.docx`:

| Variável | Numerador | Denominador |
|---|---|---|
| `pct_agua_inad` | V00112 a V00118 | V01042 |
| `pct_esgoto_inad` | V00312 a V00316 | V01042 |
| `pct_lixo_inad` | V00398 a V00402 | V01042 |
| `razao_moradores` | V00005 + V00006 | V01042 |
| `pct_analfab` | V00901 | V00900 (pop. 15+) |
| `renda_media` | V06004 (direto) | — |
| `pct_raca_pretpardind` | V01318 + V01320 + V01321 | v0001 |

**Tratamento de sigilo (decisão para EDA):** se *todas* as variáveis-numerador de
um indicador estão sigilosas (`NaN`), o indicador vira `NaN` também — não zero.
Isto preserva a transparência sobre dados faltantes nas tabelas descritivas e nos
mapas de missing. Implementado via `sum(axis=1, min_count=1)`.

> Decisão diferente da Fase 2, que convertia sigilo residual para zero (apropriado
> para o cálculo final do índice, mas mascara dados faltantes durante a EDA).

In [ ]:
def safe_div(num, den):
    return np.where((den > 0) & den.notna(), num / den, np.nan)

agua_cols   = ['V00112', 'V00113', 'V00114', 'V00115', 'V00116', 'V00117', 'V00118']
esgoto_cols = ['V00312', 'V00313', 'V00314', 'V00315', 'V00316']
lixo_cols   = ['V00398', 'V00399', 'V00400', 'V00401', 'V00402']
raca_cols   = ['V01318', 'V01320', 'V01321']

# min_count=1 → se TODAS as parcelas estiverem sigilosas (NaN), a soma vira NaN
# (em vez do default 0). Isto preserva o sigilo nas proporções.
df_ok['pct_agua_inad']        = safe_div(df_ok[agua_cols].sum(axis=1, min_count=1),   df_ok['V01042'])
df_ok['pct_esgoto_inad']      = safe_div(df_ok[esgoto_cols].sum(axis=1, min_count=1), df_ok['V01042'])
df_ok['pct_lixo_inad']        = safe_div(df_ok[lixo_cols].sum(axis=1, min_count=1),   df_ok['V01042'])
df_ok['razao_moradores']      = safe_div(df_ok[['V00005','V00006']].sum(axis=1, min_count=1), df_ok['V01042'])
df_ok['pct_analfab']          = safe_div(df_ok['V00901'], df_ok['V00900'])
df_ok['renda_media']          = df_ok['V06004']
df_ok['pct_raca_pretpardind'] = safe_div(df_ok[raca_cols].sum(axis=1, min_count=1), df_ok['v0001'])

INDICADORES = ['pct_agua_inad', 'pct_esgoto_inad', 'pct_lixo_inad',
               'razao_moradores', 'pct_analfab', 'renda_media',
               'pct_raca_pretpardind']

for c in [x for x in INDICADORES if x.startswith('pct_')]:
    df_ok[c] = df_ok[c].clip(lower=0, upper=1)

print('Proporções calculadas. Resumo rápido:')
print(df_ok[INDICADORES].describe().round(4).to_string())

## 5. Descritivas globais das 7 variáveis

Tabela síntese seguindo o guia FIOCRUZ (Seções 3 e 4): n, média, DP, CV, mínimo,
quartis, máximo, IQR, assimetria e curtose. Para o artigo, o par **(mediana, IQR)**
é mais robusto que **(média, DP)** quando há assimetria/outliers — calculamos os
dois para escolher na hora da redação.

In [ ]:
def descritiva(s):
    s = s.dropna()
    if len(s) == 0:
        return pd.Series({k: np.nan for k in
            ['n','media','dp','cv_pct','min','p25','mediana','p75','max','iq','assim','curt']})
    mean = s.mean()
    return pd.Series({
        'n': int(len(s)),
        'media':   mean,
        'dp':      s.std(),
        'cv_pct':  (s.std() / mean * 100) if mean != 0 else np.nan,
        'min':     s.min(),
        'p25':     s.quantile(0.25),
        'mediana': s.median(),
        'p75':     s.quantile(0.75),
        'max':     s.max(),
        'iq':      s.quantile(0.75) - s.quantile(0.25),
        'assim':   s.skew(),
        'curt':    s.kurtosis(),
    })

desc_global = pd.DataFrame({c: descritiva(df_ok[c]) for c in INDICADORES}).T
desc_global = desc_global.round(4)
print('Descritivas globais (setores OK dos 70 municípios ELSI):\n')
print(desc_global.to_string())

## 6. Descritivas por município

Tabela longa com, para cada município e variável: `n`, média, DP, mediana, P25, P75.
Útil para a **Tabela 1 do artigo** e para identificar municípios com perfis
discrepantes.

In [ ]:
def desc_grupo(grupo, col):
    s = grupo[col].dropna()
    if len(s) == 0:
        return pd.Series({k: np.nan for k in ['n','media','dp','p25','mediana','p75']})
    return pd.Series({
        'n':       int(len(s)),
        'media':   s.mean(),
        'dp':      s.std(),
        'p25':     s.quantile(0.25),
        'mediana': s.median(),
        'p75':     s.quantile(0.75),
    })

linhas = []
for (cd_uf, cd_mun, nm_mun), g in df_ok.groupby(['CD_UF', 'CD_MUN', 'NM_MUN'], sort=True):
    for col in INDICADORES:
        d = desc_grupo(g, col)
        d['CD_UF'] = cd_uf
        d['CD_MUN'] = cd_mun
        d['NM_MUN'] = nm_mun
        d['variavel'] = col
        linhas.append(d)
desc_mun = pd.DataFrame(linhas)[['CD_UF','CD_MUN','NM_MUN','variavel','n','media','dp','p25','mediana','p75']]
desc_mun = desc_mun.sort_values(['CD_UF','NM_MUN','variavel']).reset_index(drop=True)
print(f'Linhas geradas: {len(desc_mun)} (70 municípios × {len(INDICADORES)} variáveis)')
print('\nPrimeiras 14 linhas (2 municípios × 7 variáveis):')
print(desc_mun.head(14).round(4).to_string(index=False))

## 7. Descritivas por região geográfica

Agrega o resultado por região (Norte / Nordeste / Sudeste / Sul / Centro-Oeste)
usando a lista oficial ELSI como dicionário UF → região.

In [ ]:
df_elsi = pd.read_csv(str(ROOT / 'dados' / 'municipios_elsi_brasil.csv'), sep=';', dtype=str)
mapa_regiao = dict(zip(df_elsi['uf_codigo'].str.zfill(2), df_elsi['regiao']))
df_ok['regiao'] = df_ok['CD_UF'].map(mapa_regiao)

linhas = []
for regiao, g in df_ok.groupby('regiao', sort=True):
    for col in INDICADORES:
        d = desc_grupo(g, col)
        d['regiao'] = regiao
        d['variavel'] = col
        linhas.append(d)
desc_reg = pd.DataFrame(linhas)[['regiao','variavel','n','media','dp','p25','mediana','p75']]
desc_reg = desc_reg.sort_values(['regiao','variavel']).reset_index(drop=True)
print(desc_reg.round(4).to_string(index=False))

## 8. Distribuições — histogramas

Histograma de cada uma das 7 variáveis (guia FIOCRUZ, Seção 5.5). Permite ver a
forma da distribuição: simétrica, assimétrica, bimodal, com massa em zero, etc.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(INDICADORES):
    ax = axes[i]
    df_ok[col].dropna().hist(bins=50, ax=ax, color='#4C72B0', edgecolor='white')
    ax.set_title(col, fontsize=10)
    ax.set_xlabel(''); ax.set_ylabel('frequência')
    ax.grid(False)
for j in range(len(INDICADORES), len(axes)):
    axes[j].axis('off')
fig.suptitle('Histogramas — 7 variáveis-componente do IVS (setores OK, 70 municípios ELSI)', fontsize=12)
fig.tight_layout()
fig.savefig(CAMINHO_FIG + 'histogramas.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Distribuições — boxplots por região

Boxplots estratificados por região (guia FIOCRUZ, Seção 5.3) para comparar centro,
dispersão e outliers entre regiões.

In [ ]:
ordem_regiao = ['Norte','Nordeste','Sudeste','Sul','Centro-Oeste']
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()
for i, col in enumerate(INDICADORES):
    ax = axes[i]
    dados = [df_ok[df_ok['regiao'] == r][col].dropna() for r in ordem_regiao]
    ax.boxplot(dados, tick_labels=ordem_regiao, showfliers=True,
               medianprops={'color': 'red'})
    ax.set_title(col, fontsize=10)
    ax.tick_params(axis='x', rotation=30)
    ax.grid(False)
for j in range(len(INDICADORES), len(axes)):
    axes[j].axis('off')
fig.suptitle('Boxplots por região — variáveis-componente do IVS', fontsize=12)
fig.tight_layout()
fig.savefig(CAMINHO_FIG + 'boxplots_por_regiao.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Análise de outliers (regra do IQR)

Conta o número de setores fora do intervalo `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]` para cada
variável (guia FIOCRUZ, Seção 5.3). Outliers não devem ser removidos automaticamente
nesta fase — apenas inventariados.

In [ ]:
def conta_outliers(s):
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iq = q3 - q1
    lim_inf, lim_sup = q1 - 1.5*iq, q3 + 1.5*iq
    fora = ((s < lim_inf) | (s > lim_sup)).sum()
    return pd.Series({
        'n_validos':   int(len(s)),
        'q1': q1, 'q3': q3, 'iq': iq,
        'lim_inf': lim_inf, 'lim_sup': lim_sup,
        'n_outliers':  int(fora),
        'pct_outliers': round(fora / len(s) * 100, 2) if len(s) else np.nan,
    })

outliers = pd.DataFrame({c: conta_outliers(df_ok[c]) for c in INDICADORES}).T
print('Outliers por variável (regra IQR — 1.5 × IQR):\n')
print(outliers.round(4).to_string())

## 11. Análise de dados faltantes

Mapa de calor da % de células faltantes em cada variável-componente, por município.
Indispensável para detectar municípios com cobertura ruim (guia FIOCRUZ, checklist
20.2).

In [ ]:
miss_mun = (df_ok.groupby('NM_MUN')[INDICADORES]
            .apply(lambda g: g.isna().mean() * 100)
            .round(2))
print('Top 10 municípios com mais dados faltantes (média entre variáveis):')
miss_mun['_media'] = miss_mun.mean(axis=1)
print(miss_mun.sort_values('_media', ascending=False).head(10).to_string())

fig, ax = plt.subplots(figsize=(8, max(8, len(miss_mun) * 0.15)))
m = miss_mun.drop(columns='_media').values
im = ax.imshow(m, aspect='auto', cmap='Reds', vmin=0, vmax=max(1, m.max()))
ax.set_yticks(range(len(miss_mun)))
ax.set_yticklabels(miss_mun.index, fontsize=6)
ax.set_xticks(range(len(INDICADORES)))
ax.set_xticklabels(INDICADORES, rotation=45, ha='right', fontsize=8)
fig.colorbar(im, ax=ax, label='% faltante')
ax.set_title('Dados faltantes (%) por município × variável')
fig.tight_layout()
fig.savefig(CAMINHO_FIG + 'missing_por_municipio.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Matriz de correlação

Correlação de Pearson (linear) e Spearman (postos / não paramétrica) entre as 7
variáveis (guia FIOCRUZ, Seção 11). Subsidia a futura análise fatorial — variáveis
muito correlacionadas tendem a se agrupar no mesmo fator.

In [ ]:
corr_p = df_ok[INDICADORES].corr(method='pearson').round(3)
corr_s = df_ok[INDICADORES].corr(method='spearman').round(3)

print('Pearson:\n', corr_p.to_string(), '\n')
print('Spearman:\n', corr_s.to_string())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 6))
for ax, mat, titulo in [(a1, corr_p, 'Pearson'), (a2, corr_s, 'Spearman')]:
    im = ax.imshow(mat.values, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(INDICADORES))); ax.set_yticks(range(len(INDICADORES)))
    ax.set_xticklabels(INDICADORES, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(INDICADORES, fontsize=8)
    ax.set_title(f'Correlação — {titulo}')
    for i in range(len(INDICADORES)):
        for j in range(len(INDICADORES)):
            ax.text(j, i, f'{mat.values[i,j]:.2f}', ha='center', va='center',
                    color='black' if abs(mat.values[i,j]) < 0.6 else 'white', fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(CAMINHO_FIG + 'matriz_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Exportação dos artefatos

Salva os CSVs com as tabelas descritivas em `banco_de_dados/eda/`. As figuras
já foram salvas em `banco_de_dados/eda/figuras/` ao longo das células anteriores.

In [ ]:
desc_global.to_csv(CAMINHO_EDA + 'descritivas_globais.csv',       sep=';', encoding='utf-8-sig')
desc_mun.to_csv(   CAMINHO_EDA + 'descritivas_por_municipio.csv', sep=';', encoding='utf-8-sig', index=False)
desc_reg.to_csv(   CAMINHO_EDA + 'descritivas_por_regiao.csv',    sep=';', encoding='utf-8-sig', index=False)
outliers.to_csv(   CAMINHO_EDA + 'outliers.csv',                  sep=';', encoding='utf-8-sig')
miss_mun.drop(columns='_media').to_csv(CAMINHO_EDA + 'missing_por_municipio.csv',
                                        sep=';', encoding='utf-8-sig')
corr_p.to_csv(CAMINHO_EDA + 'correlacao_pearson.csv',  sep=';', encoding='utf-8-sig')
corr_s.to_csv(CAMINHO_EDA + 'correlacao_spearman.csv', sep=';', encoding='utf-8-sig')
resumo.to_csv(CAMINHO_EDA + 'elegibilidade_setores.csv', sep=';', encoding='utf-8-sig')

print('Artefatos exportados em', CAMINHO_EDA)
print('Figuras em', CAMINHO_FIG)